<a href="https://colab.research.google.com/github/daniswaramp/acc-heat-analysis/blob/main/ACC_Heat_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accelerated Cooling (ACC) of hot-rolled plates

## Empirical calculation and a 3-D finite difference heat transfer analysis

This notebook follows a hot-rolled steel plate from the finishing mill to the cooling bed and answers one question: how different are the temperatures at the surface, at quarter depth (1/4), at one-third depth (1/3) and at the core, during and after accelerated cooling?

The journey:

1. The plate leaves the finish rolling mill at the finish rolling temperature (FRT).
2. It travels down the run-out table at the table run speed, passing under laminar water curtains, `n_cur` headers on top and `n_cur` on the bottom, spaced `s_gap` apart along the table.
3. Each curtain impinges a strip of cooling water onto the plate. Between curtains, and after the last one, the plate cools slowly by air convection and radiation.
4. Water cooling is fierce, of order thousands of W/m²K, while steel conducts heat slowly (α ≈ 6×10⁻⁶ m²/s), so the surface cools far faster than the core and a through-thickness gradient builds.
5. An empirical module pre-calculates the water flux, the boiling-regime heat transfer coefficient, the Biot number and three analytical estimates: lumped capacitance, semi-infinite solid and Heisler one-term.
6. A 3-D explicit finite difference (FDM) solver resolves the full transient temperature field `T(thickness, width, length, t)` with the moving-curtain boundary condition, and slices it in every direction.

Run all cells top to bottom. The notebook needs only `numpy` and `matplotlib`, and Colab has both already.

## 1 · The physics: transient heat conduction

### 1.1 Fourier's law of conduction

$$q = -k\,\nabla T$$

### 1.2 The heat diffusion equation (3-D, constant properties)

$$\rho\,c_p\,\frac{\partial T}{\partial t}
= k\left(\frac{\partial^2 T}{\partial x^2}+\frac{\partial^2 T}{\partial y^2}+\frac{\partial^2 T}{\partial z^2}\right)
\quad\Longleftrightarrow\quad
\frac{\partial T}{\partial t}=\alpha\,\nabla^{2}T,
\qquad \alpha=\frac{k}{\rho c_p}$$

with the plate axes x = thickness, y = width, z = length (rolling direction).

### 1.3 Boundary conditions

Convection, Newton's law of cooling, applies under the water curtains ($T_f = T_w$) and in air ($T_f = T_\infty$):

$$-k\,\frac{\partial T}{\partial n}\bigg|_{s} = h\,\left(T_s - T_f\right)$$

Radiation (Stefan-Boltzmann) matters on hot air-exposed surfaces:

$$q_{rad}=\varepsilon\,\sigma\left(T_s^{4}-T_f^{4}\right),
\qquad \sigma = 5.670\times10^{-8}\ \mathrm{W/m^2K^4}$$

The two combine into an effective film coefficient:

$$h_{eff}=h_{conv}+\varepsilon\,\sigma\left(T_s^{2}+T_f^{2}\right)\left(T_s+T_f\right)$$

The symmetry plane at the plate centre-line in width carries zero flux, $\partial T/\partial n = 0$, so only half the width needs to be modelled.

### 1.4 Dimensionless numbers

$$\text{Biot: } Bi=\frac{h\,(t_h/2)}{k}
\qquad\qquad
\text{Fourier: } Fo=\frac{\alpha\,t}{(t_h/2)^2}$$

Below $Bi = 0.1$ the plate behaves almost as a lumped system. For $Bi \gg 0.1$ the internal gradients are strong and the full finite difference solution is required.

### 1.5 Analytical solutions (benchmarks for the FDM)

Lumped capacitance, exact for $Bi<0.1$:

$$\frac{T-T_\infty}{T_i-T_\infty}=\exp\!\left(-\frac{t}{\tau}\right)=\exp(-Bi\cdot Fo),
\qquad \tau=\frac{\rho c_p V}{h A}=\frac{\rho c_p (t_h/2)}{h}$$

Semi-infinite solid with a convective surface, which gives the surface response during the short first curtain contact ($\beta = \tfrac{h}{k}\sqrt{\alpha t}$):

$$\frac{T_s-T_\infty}{T_i-T_\infty}
=\exp\!\left(\frac{h^{2}\alpha t}{k^{2}}\right)\mathrm{erfc}\!\left(\frac{h\sqrt{\alpha t}}{k}\right)$$

Heisler one-term approximation, for the mid-plane of a plate suddenly exposed to $h$ on both faces, with $\zeta_1\tan\zeta_1 = Bi$ and $C_1 = \dfrac{4\sin\zeta_1}{2\zeta_1+\sin 2\zeta_1}$:

$$\frac{\theta_m}{\theta_i}=C_1\,\exp\!\left(-\zeta_1^{2}\,Fo\right)$$

### 1.6 Empirical correlations

Water-flux heat transfer coefficient for laminar curtain cooling of hot steel, in power-law form with constants taken from the ACC literature:

$$h_w = C\,\dot{W}^{\,n},\qquad
\dot{W}=\frac{\dot{Q}_{header}}{W\cdot l_{imp}}\ \ [\mathrm{m^3/m^2 s}]$$

Boiling regimes, an optional toggle: above the Leidenfrost temperature the surface is blanketed by stable film boiling (low $h$). Below it the water wets the surface and transition or nucleate boiling gives much higher $h$.

Cooling rate:

$$CR=-\frac{dT}{dt}\ \ [\mathrm{^\circ C/s}]$$

## 2 · Numerical method: explicit finite differences (FTCS)

Interior node, a 3-D second difference:

$$T_{i,j,k}^{p+1}=T_{i,j,k}^{p}
+\alpha\,\Delta t\left[
\frac{T_{i+1,j,k}^{p}-2T_{i,j,k}^{p}+T_{i-1,j,k}^{p}}{\Delta x^{2}}
+\frac{T_{i,j+1,k}^{p}-2T_{i,j,k}^{p}+T_{i,j-1,k}^{p}}{\Delta y^{2}}
+\frac{T_{i,j,k+1}^{p}-2T_{i,j,k}^{p}+T_{i,j,k-1}^{p}}{\Delta z^{2}}\right]$$

Surface node, from an energy balance on a half-cell ($T_{in}$ is the first interior node):

$$\rho c_p\,\frac{\Delta x}{2}\,\frac{dT_s}{dt}
=\frac{2k}{\Delta x}\left(T_{in}-T_s\right)+h\left(T_f-T_s\right)$$

Stability criterion for the explicit scheme, with a 0.8 safety factor:

$$\Delta t \;\le\; 0.8\cdot\frac{1}{2\alpha\left(\dfrac{1}{\Delta x^{2}}+\dfrac{1}{\Delta y^{2}}+\dfrac{1}{\Delta z^{2}}\right)}$$

Moving-curtain schedule in the Lagrangian frame. The plate stays fixed on the grid and header $j$ sweeps along it at the run speed. Its position at time $t$ is

$$c_j(t)=v\,t-(j-1)\,s_{gap}$$

and any surface point $x$ is water-cooled whenever $|x-c_j(t)|\le l_{imp}/2$ for some $j$. Everywhere else the surface sees air convection and radiation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import math, os
from math import erfc, pi, tan
from IPython.display import Markdown, display, HTML

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 11, "figure.facecolor": "white"})
FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)
print("numpy", np.__version__, "| matplotlib", mpl.__version__)

## 3 · Inputs, the only cell you need to edit

These are the process constraints: plate geometry, header count per side (`n_cur = 1` means 1 top and 1 bottom), table run speed, finish rolling temperature and the uniform header spacing. The next cell holds everything else fixed. The `tune_factor` rescales the water-side HTC, so keep it at 1.0 for the pure prediction, or calibrate it (section 6b) when the line behaves more or less severely than the correlation suggests.

In [ ]:
# =====================================================================
#  USER-ADJUSTABLE INPUTS
# =====================================================================
t_h    = 25.0      # plate thickness                                [mm]
W      = 2000.0    # plate width                                    [mm]
L      = 12000.0   # plate length                                   [mm]
n_cur  = 5         # header curtain number (per side: 5 -> 5 top + 5 bottom)
v      = 60.0      # table run speed during cooling                 [m/min]
FRT    = 880.0     # finish rolling temperature (before laminar)    [degC]
s_gap  = 3000.0    # header spacing (uniform gap between headers)   [mm]
tune_factor = 1.0  # HTC tuning factor: >1 = more severe cooling than predicted,
                    #                   <1 = less severe than predicted  [-]

print(f"Plate {t_h:.0f} x {W:.0f} x {L:.0f} mm | {n_cur} top + {n_cur} bottom curtains | "
      f"{v:.0f} m/min | FRT {FRT:.0f} degC | spacing {s_gap:.0f} mm | tune_factor {tune_factor:.2f}")

## 4 · Fixed parameters

Material, fluid, header design and ambient conditions are fixed.

In [ ]:
# =====================================================================
#  FIXED PARAMETERS
# =====================================================================
k        = 30.0      # thermal conductivity of steel           [W/m/K]
rho      = 7850.0    # density                                 [kg/m^3]
cp       = 650.0     # specific heat                           [J/kg/K]
Tw       = 30.0      # cooling water temperature               [degC]
Tinf     = 30.0      # ambient air temperature                 [degC]
eps      = 0.8       # surface emissivity                      [-]
SIGMA    = 5.670e-8  # Stefan-Boltzmann constant               [W/m^2/K^4]
h_air    = 20.0      # air convection HTC (gaps + after exit)  [W/m^2/K]
l_imp    = 250.0     # effective curtain impingement length    [mm]
Q_header = 2000.0    # water flow rate per header              [L/min]
FCT_tgt  = 600.0     # target finish cooling temperature       [degC]
t_post   = 30.0      # post-ACC air cooling time to simulate   [s]

# computational grid refinement
n_t, n_w, n_l = 26, 201, 241     # nodes across thickness / half-width / length

# optional boiling-regime HTC toggle (film boiling vs transition/nucleate)
boiling_regime = False
T_leid = 350.0     # Leidenfrost temperature [degC]
h_film = 1200.0    # film boiling HTC        [W/m^2/K]
h_nuc  = 8000.0    # transition/nucleate HTC [W/m^2/K]

alpha = k / (rho * cp)
display(Markdown(f"""
| Parameter | Value | Unit |
|---|---|---|
| Thermal conductivity k | {k} | W/m/K |
| Density rho | {rho} | kg/m3 |
| Specific heat cp | {cp} | J/kg/K |
| Thermal diffusivity alpha | {alpha:.3e} | m2/s |
| Water / ambient temperature | {Tw} | degC |
| Emissivity | {eps} | - |
| Air HTC (between curtains) | {h_air} | W/m2/K |
| Curtain impingement length | {l_imp:.0f} | mm |
| Water flow per header | {Q_header:.0f} | L/min |
| Target FCT (core) | {FCT_tgt} | degC |
| Post-ACC air cooling time | {t_post} | s |
"""))

## 5 · The run-out table at a glance

The plate (dark slab) leaves the mill and sweeps under the top curtains (blue) while the bottom curtains spray from below. Header $j$ meets the plate leading edge at $t=(j-1)\,s_{gap}/v$. In the solver the plate stays fixed and the curtains sweep over it.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
v_ms, s_gap_m = v / 60.0, s_gap * 1e-3
plate_len = L * 1e-3
first_hdr = 2.0                                    # schematic position of header 1 [m]
hdr_x = [first_hdr + j * s_gap_m for j in range(n_cur)]

# run-out table rollers
for xr in np.arange(-1.0, hdr_x[-1] + 1.5, 1.0):
    ax.add_patch(plt.Circle((xr, -0.55), 0.12, fc="0.75", ec="0.4", zorder=1))
ax.plot([-1.8, hdr_x[-1] + 2.0], [-0.30, -0.30], color="0.4", lw=2)

# plate (leading edge at header 1 at t = 0)
ax.add_patch(plt.Rectangle((hdr_x[0] - plate_len, -0.30), plate_len, 0.35,
                           fc="0.25", ec="k", zorder=3))
ax.annotate(f"plate {t_h:.0f} mm thick (exaggerated)", xy=(hdr_x[0] - plate_len / 2, -0.12),
            ha="center", color="white", fontsize=9, zorder=4)

# top and bottom curtains
for hx in hdr_x:
    ax.add_patch(plt.Rectangle((hx - 0.12, 0.35), 0.24, 0.55, fc="tab:blue", ec="none", alpha=0.85))
    ax.annotate("", xy=(hx, -0.02), xytext=(hx, 0.32),
                arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=2))
    ax.add_patch(plt.Rectangle((hx - 0.12, -1.15), 0.24, 0.45, fc="tab:blue", ec="none", alpha=0.45))
    ax.annotate("", xy=(hx, -0.33), xytext=(hx, -0.68),
                arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=1.5, alpha=0.6))

# annotations
ax.annotate("MILL EXIT / FRT", xy=(-1.6, 0.15), fontsize=10, fontweight="bold", color="darkred")
ax.annotate("", xy=(hdr_x[0] + 0.8, -1.5), xytext=(hdr_x[0] - 0.8, -1.5),
            arrowprops=dict(arrowstyle="-|>", color="crimson", lw=3))
ax.annotate(f"run speed {v:.0f} m/min", xy=(hdr_x[0], -1.85), ha="center", color="crimson")
ax.annotate("", xy=(hdr_x[1], 1.05), xytext=(hdr_x[0], 1.05),
            arrowprops=dict(arrowstyle="<->", color="k"))
ax.annotate(f"header spacing $s_{{gap}}$ = {s_gap:.0f} mm", xy=((hdr_x[0] + hdr_x[1]) / 2, 1.15),
            ha="center", fontsize=10)
for j, hx in enumerate(hdr_x):
    ax.annotate(f"H{j+1}", xy=(hx, 1.0), ha="center", color="tab:blue", fontweight="bold")
ax.set_xlim(-2.2, hdr_x[-1] + 2.2); ax.set_ylim(-2.2, 1.6)
ax.set_title(f"Laminar ACC line: {n_cur} top + {n_cur} bottom curtains", fontsize=13)
ax.axis("off"); fig.tight_layout()
fig.savefig(f"{FIGDIR}/schematic.png", dpi=150)
plt.show()

## 6 · Empirical pre-calculation

The empirical module sizes the cooling before the FDM runs: water flux first, then the heat transfer coefficient, then the Biot number, then the analytical time estimates. These set the FDM boundary condition and also provide its benchmarks.

In [ ]:
# =====================================================================
#  EMPIRICAL PRE-CALCULATION
# =====================================================================
l_imp_m, t_h_m = l_imp * 1e-3, t_h * 1e-3

# --- 1. water flux and empirical HTC (power-law water-flux correlation) ---
W_flux   = (Q_header / 60.0) / (W * 1e-3 * l_imp_m)   # [L/m2/s]
W_flux_m = W_flux / 1000.0                            # [m3/m2/s]
C_corr, n_corr = 1.2e4, 0.45                          # correlation constants
h_w = C_corr * W_flux_m ** n_corr                     # [W/m2/K]
h_w_tuned = tune_factor * h_w                         # tuned HTC actually applied

# --- 2. Biot number ---
Bi = h_w_tuned * (t_h_m / 2) / k

# --- 3. lumped capacitance (exact only for Bi < 0.1) ---
tau_lump = rho * cp * (t_h_m / 2) / h_w_tuned         # time constant [s]
t_lump   = tau_lump * np.log((FRT - Tw) / (FCT_tgt - Tw))   # time to FCT target [s]

# --- 4. semi-infinite solid: surface response during curtain contact ---
def Ts_semiinf(t):
    b = h_w_tuned * np.sqrt(alpha * t) / k
    return FRT + (Tw - FRT) * (1.0 - np.exp(b ** 2) * erfc(b))
t_exp   = l_imp_m / v_ms                              # one curtain exposure time [s]
Ts_1hdr = Ts_semiinf(t_exp)

# --- 5. Heisler one-term approximation: solve zeta*tan(zeta) = Bi ---
lo, hi = 1e-9, pi / 2 - 1e-9
for _ in range(200):
    mid = 0.5 * (lo + hi)
    if mid * tan(mid) > Bi: hi = mid
    else: lo = mid
zeta1 = 0.5 * (lo + hi)
C1 = 4 * np.sin(zeta1) / (2 * zeta1 + np.sin(2 * zeta1))

print(f"Water flux                 : {W_flux:8.2f} L/m2/s")
print(f"Empirical HTC  h_w         : {h_w:8.0f} W/m2/K")
print(f"Tuning factor              : {tune_factor:8.2f}  -> applied HTC {h_w_tuned:.0f} W/m2/K")
print(f"Biot number    Bi          : {Bi:8.3f}  -> " +
      ("lumped OK (Bi < 0.1)" if Bi < 0.1 else "NOT lumped: internal gradients matter, FDM required"))
print(f"Lumped time to {FCT_tgt:.0f} degC      : {t_lump:8.2f} s")
print(f"One curtain exposure       : {t_exp:8.2f} s -> surface {FRT:.0f} -> {Ts_1hdr:.0f} degC (semi-infinite)")
print(f"Heisler  zeta1={zeta1:.4f}, C1={C1:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ts = np.linspace(0, t_exp, 100)
ax1.plot(ts * 1e3, [Ts_semiinf(x) for x in ts], lw=2.5, color="tab:blue")
ax1.axhline(Tw, color="tab:cyan", ls="--", label="water temperature")
ax1.set_xlabel("exposure time under curtain [ms]"); ax1.set_ylabel("surface temperature [degC]")
ax1.set_title("Semi-infinite solid surface response"); ax1.legend()

tl = np.linspace(0, t_lump, 200)
ax2.plot(tl, Tw + (FRT - Tw) * np.exp(-tl / tau_lump), lw=2.5, color="tab:red",
         label=f"lumped, tau = {tau_lump:.1f} s")
ax2.axhline(FCT_tgt, color="k", ls="--", label=f"FCT target {FCT_tgt:.0f} degC")
ax2.set_xlabel("time [s]"); ax2.set_ylabel("bulk temperature [degC]")
ax2.set_title("Lumped-capacitance cooling curve"); ax2.legend()
fig.tight_layout(); fig.savefig(f"{FIGDIR}/empirical.png", dpi=150)
plt.show()

## 6b · Tuning factor: matching the model to plant reality

Real ACC lines rarely behave exactly as the textbook correlations predict. Surface scale, water pressure and flux distribution across the header, curtain alignment, plate roughness and recalescence all shift the effective cooling severity. The tuning factor rescales every water-side heat transfer coefficient:

$$h_{w,\,eff} = \mathrm{tune\_factor} \times h_w$$

Setting `tune_factor` above 1 means the line cools more severely than predicted. Setting it below 1 means it cools less severely.

If you have a measured finish cooling temperature, from a core thermocouple or an exit pyrometer, the calibration tool below finds the tuning factor that reproduces it. It runs a fast 1-D through-thickness solver with the same physics and the same curtain schedule at the plate centre, so the search takes seconds rather than a full 3-D run per trial.

In [ ]:
# =====================================================================
#  TUNING FACTOR CALIBRATION (fast 1-D through-thickness solver)
# =====================================================================
def solve_core_1d(factor, n1d=51):
    """Core temperature at end of ACC - 1-D through thickness, plate-centre schedule."""
    dx1 = t_h * 1e-3 / (n1d - 1)
    dt1 = 0.8 / (2 * alpha / dx1**2)
    h1  = factor * h_w
    v1  = v / 60.0
    t_end1 = (L * 1e-3 + (n_cur - 1) * s_gap * 1e-3 + l_imp * 1e-3) / v1
    x_c = L * 1e-3 / 2
    sg, lim = s_gap * 1e-3, l_imp * 1e-3
    t0s = np.array([(x_c + j * sg) / v1 for j in range(n_cur)])   # curtain windows at centre
    T1 = np.full(n1d, FRT)
    t = 0.0
    while t < t_end1:
        in_water = np.any((t0s <= t) & (t <= t0s + lim / v1))
        h_rad = eps * SIGMA * (T1**2 + Tinf**2) * (T1 + Tinf)
        h_face = h1 if in_water else h_air + h_rad[0]
        Tn = T1.copy()
        Tn[1:-1] = T1[1:-1] + dt1 * alpha * (T1[2:] - 2 * T1[1:-1] + T1[:-2]) / dx1**2
        Tn[0]  = T1[0]  + dt1 * (2 * alpha / dx1**2 * (T1[1]  - T1[0])  + 2 * h_face / (rho * cp * dx1) * (Tw - T1[0]))
        Tn[-1] = T1[-1] + dt1 * (2 * alpha / dx1**2 * (T1[-2] - T1[-1]) + 2 * h_face / (rho * cp * dx1) * (Tw - T1[-1]))
        T1 = Tn
        t += dt1
    return T1[(n1d - 1) // 2]

FCT_measured = 780.0    # <-- replace with your plant measurement (core FCT at ACC exit, degC)

def calibrate(FCT_meas, lo=0.1, hi=5.0, tol=1e-3):
    """Bisection on the tuning factor to match a measured core FCT (monotonic: higher factor = cooler)."""
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if solve_core_1d(mid) > FCT_meas: lo = mid
        else: hi = mid
        if hi - lo < tol: break
    return 0.5 * (lo + hi)

f_cal = calibrate(FCT_measured)
print(f"Predicted core FCT (factor = 1.00) : {solve_core_1d(1.0):.1f} degC")
print(f"Measured core FCT                  : {FCT_measured:.1f} degC")
print(f"Calibrated tuning factor           : {f_cal:.3f}")
print(f"Check with calibrated factor       : {solve_core_1d(f_cal):.1f} degC")
print("-> set tune_factor = {:.3f} in the inputs cell for the full 3-D run".format(f_cal))

fs = np.linspace(0.2, 3.0, 15)
fcts = [solve_core_1d(f) for f in fs]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(fs, fcts, "o-", lw=2, label="1-D model response")
ax.axhline(FCT_measured, color="crimson", ls="--", label=f"measured FCT {FCT_measured:.0f} degC")
ax.axvline(f_cal, color="crimson", ls=":")
ax.plot(f_cal, FCT_measured, "s", color="crimson", ms=9, label=f"calibrated factor = {f_cal:.2f}")
ax.axvline(1.0, color="gray", ls=":", label="uncalibrated (factor = 1)")
ax.set_xlabel("Tuning factor [-]"); ax.set_ylabel("Core temperature at end of ACC [degC]")
ax.set_title("Cooling severity response - tuning factor sweep"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIGDIR}/tuning.png", dpi=150)
plt.show()

## 7 · The 3-D finite difference simulation

Half the width is modelled, by symmetry. The curtains sweep over the fixed plate. Under each curtain the surface sees $h_w$, and everywhere else it sees air convection and radiation. The solver samples the four depths at the plate centre and stores full-field snapshots for the contour maps.

In [ ]:
# =====================================================================
#  3-D EXPLICIT FTCS TRANSIENT CONDUCTION SOLVER
#  Lagrangian frame: plate fixed, curtains sweep over it at speed v
# =====================================================================

# ---------- computational grid (symmetry: half width) ----------
dx_t = t_h * 1e-3 / (n_t - 1)
dy_w = W * 1e-3 / 2 / (n_w - 1)
dx_l = L * 1e-3 / (n_l - 1)
x_t = np.linspace(0, t_h * 1e-3, n_t)      # depth from TOP surface [m]
y_w = np.linspace(0, W * 1e-3 / 2, n_w)    # side edge -> centreline [m]
x_l = np.linspace(0, L * 1e-3, n_l)        # leading -> trailing edge [m]

# ---------- time step from the stability criterion ----------
dt_cond = 1.0 / (2 * alpha * (1 / dx_t**2 + 1 / dy_w**2 + 1 / dx_l**2))
h_max   = max(h_w_tuned, h_film * tune_factor if boiling_regime else 0.0)
dt_surf = 1.0 / (2 * alpha / dx_t**2 + 2 * h_max / (rho * cp * dx_t))
dt = 0.8 * min(dt_cond, dt_surf)

# ---------- moving-curtain schedule ----------
v_ms = v / 60.0
hdr_shift = np.arange(n_cur) * s_gap * 1e-3            # (j-1)*s_gap [m]
l_imp_m = l_imp * 1e-3

def water_mask(t):
    """length positions of the plate currently under ANY curtain"""
    c = v_ms * t - hdr_shift                           # curtain positions along plate [m]
    return (np.abs(x_l[None, :] - c[:, None]) <= l_imp_m / 2).any(axis=0)

# ---------- timeline ----------
t_acc_end = (L * 1e-3 + (n_cur - 1) * s_gap * 1e-3 + l_imp_m) / v_ms
t_end     = t_acc_end + t_post
n_steps   = int(np.ceil(t_end / dt))

# ---------- sampling locations ----------
i_q  = int(round((n_t - 1) * 0.25))     # quarter depth
i_t3 = int(round((n_t - 1) / 3))        # one-third depth
i_c  = (n_t - 1) // 2                   # core
j_c, k_c = (n_w - 1) // 2, (n_l - 1) // 2

T = np.full((n_t, n_w, n_l), FRT)
t_hist = [0.0]
Ts_h, Tq_h, T3_h, Tc_h = [FRT], [FRT], [FRT], [FRT]
snap_targets = [f * t_acc_end for f in (0.25, 0.5, 0.75, 1.0)] + [t_end]
snaps, snap_idx = [], 0
frame_every = max(1, n_steps // 40)
frames = [(0.0, T[:, j_c, :].copy())]

print(f"Grid {n_t} x {n_w} x {n_l} nodes | dt = {dt:.4f} s | {n_steps} steps "
      f"(ACC ends at {t_acc_end:.2f} s) | h_w applied = {h_w_tuned:.0f} W/m2/K (tune {tune_factor:.2f})")
for step in range(1, n_steps + 1):
    t = step * dt
    mask = water_mask(t)
    h_rad = eps * SIGMA * (T**2 + Tinf**2) * (T + Tinf)   # linearised radiation

    # --- second differences (one-sided at faces, zero-flux at symmetry) ---
    d2x = np.empty_like(T)
    d2x[1:-1] = (T[2:] - 2 * T[1:-1] + T[:-2]) / dx_t**2
    d2x[0]    = 2 * (T[1]  - T[0])  / dx_t**2
    d2x[-1]   = 2 * (T[-2] - T[-1]) / dx_t**2
    d2y = np.empty_like(T)
    d2y[:, 1:-1, :] = (T[:, 2:] - 2 * T[:, 1:-1] + T[:, :-2]) / dy_w**2
    d2y[:, 0, :]    = 2 * (T[:, 1]  - T[:, 0])  / dy_w**2
    d2y[:, -1, :]   = 2 * (T[:, -2] - T[:, -1]) / dy_w**2
    d2z = np.empty_like(T)
    d2z[:, :, 1:-1] = (T[:, :, 2:] - 2 * T[:, :, 1:-1] + T[:, :, :-2]) / dx_l**2
    d2z[:, :, 0]    = 2 * (T[:, :, 1]  - T[:, :, 0])  / dx_l**2
    d2z[:, :, -1]   = 2 * (T[:, :, -2] - T[:, :, -1]) / dx_l**2

    Tn = T + dt * alpha * (d2x + d2y + d2z)

    # --- convection + radiation source terms at the faces ---
    hw = np.where(T[0] > T_leid, h_film, h_nuc) * tune_factor if boiling_regime else h_w_tuned
    h_top = np.where(mask[None, :], hw, h_air + h_rad[0])
    h_bot = np.where(mask[None, :], hw, h_air + h_rad[-1])
    Tn[0]       += dt * 2 / (rho * cp * dx_t) * h_top * (Tw - T[0])
    Tn[-1]      += dt * 2 / (rho * cp * dx_t) * h_bot * (Tw - T[-1])
    Tn[:, -1, :] += dt * 2 / (rho * cp * dy_w) * (h_air + h_rad[:, -1, :]) * (Tinf - T[:, -1, :])
    Tn[:, :, 0]  += dt * 2 / (rho * cp * dx_l) * (h_air + h_rad[:, :, 0])  * (Tinf - T[:, :, 0])
    Tn[:, :, -1] += dt * 2 / (rho * cp * dx_l) * (h_air + h_rad[:, :, -1]) * (Tinf - T[:, :, -1])
    T = Tn

    t_hist.append(t)
    Ts_h.append(T[0, j_c, k_c]); Tq_h.append(T[i_q, j_c, k_c])
    T3_h.append(T[i_t3, j_c, k_c]); Tc_h.append(T[i_c, j_c, k_c])

    while snap_idx < len(snap_targets) and t >= snap_targets[snap_idx] - dt / 2:
        snaps.append((snap_targets[snap_idx], T.copy())); snap_idx += 1
    if step % frame_every == 0:
        frames.append((t, T[:, j_c, :].copy()))
    if step % max(1, n_steps // 10) == 0:
        print(f"  step {step:5d}/{n_steps}  t = {t:6.2f} s  T_core = {T[i_c, j_c, k_c]:7.1f} C")

t_hist = np.array(t_hist)
Ts_h, Tq_h, T3_h, Tc_h = map(np.array, (Ts_h, Tq_h, T3_h, Tc_h))
frames_t = np.array([f[0] for f in frames]); frames_T = np.array([f[1] for f in frames])
print(f"Done. min(T) = {T.min():.1f} C (stability ok if >= {Tw} C)")

## 8 · Temperature evolution at the plate centre

The headline plot is the four depths against time. Blue bands mark each curtain passing the plate centre. Every passage pulls the surface temperature down while the core barely moves, which drives the surface-to-core differential up. After the last curtain, air cooling lets the temperatures converge.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(t_hist, Ts_h, lw=2.5, color="tab:orange", label=f"Surface ({x_t[0]*1e3:.1f} mm)")
ax.plot(t_hist, Tq_h, lw=2,   color="tab:green",  label=f"Quarter depth ({x_t[i_q]*1e3:.1f} mm)")
ax.plot(t_hist, T3_h, lw=2,   color="tab:purple", label=f"One-third depth ({x_t[i_t3]*1e3:.1f} mm)")
ax.plot(t_hist, Tc_h, lw=2.5, color="k",          label=f"Core ({x_t[i_c]*1e3:.1f} mm)")
ax.axvline(t_acc_end, color="gray", ls="--", label="End of ACC")
x_c = x_l[k_c]
for j in range(n_cur):
    t0 = (x_c + j * s_gap * 1e-3) / v_ms
    ax.axvspan(t0, t0 + l_imp_m / v_ms, color="tab:blue", alpha=0.15)
ax.set_xlabel("Time [s]"); ax.set_ylabel("Temperature [degC]")
ax.set_title("Temperature evolution at plate centre (curtain passages shaded blue)")
ax.legend(loc="upper right"); fig.tight_layout()
fig.savefig(f"{FIGDIR}/evolution.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
cmap = plt.cm.plasma
for idx, (tsnap, fld) in enumerate(snaps):
    ax.plot(fld[:, j_c, k_c], x_t * 1e3, lw=2, color=cmap(idx / (len(snaps) - 1)),
            label=f"t = {tsnap:.1f} s")
ax.set_ylim(t_h, 0)
ax.set_xlabel("Temperature [degC]"); ax.set_ylabel("Depth from top surface [mm]")
ax.set_title("Through-thickness temperature profiles (plate centre)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{FIGDIR}/profiles.png", dpi=150)
plt.show()

## 9 · 2-D spatial temperature fields

Full-field slices of the 3-D solution at four snapshots of the cooling:

- Thickness × Width, a mid-length section. The width gradient builds from the two side edges, which lose heat to air while the curtains quench the centre.
- Thickness × Length, the centre strip. The leading edge exits the ACC first, so the cooling front sweeps along the plate with the curtain train.
- Width × Length, the top surface and mid-thickness planes. This is the bow-wave pattern an ACC line is known for.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharex=True, sharey=True)
for ax, (tsnap, fld) in zip(axes.ravel(), snaps):
    f2d = fld[:, :, k_c]                       # mid-length section (n_t x n_w)
    full = np.hstack([f2d[:, ::-1], f2d])      # mirror half width -> full width
    im = ax.imshow(full, extent=[-W / 2, W / 2, t_h, 0], aspect="auto",
                   cmap="inferno", vmin=Tw, vmax=FRT)
    ax.set_title(f"t = {tsnap:.1f} s")
fig.colorbar(im, ax=axes.ravel().tolist(), label="Temperature [degC]", shrink=0.9)
fig.suptitle("Thickness x Width temperature field (mid-length section)", fontsize=13)
fig.savefig(f"{FIGDIR}/contour_thickness_width.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharex=True, sharey=True)
for ax, (tsnap, fld) in zip(axes.ravel(), snaps):
    im = ax.imshow(fld[:, j_c, :], extent=[0, L, t_h, 0], aspect="auto",
                   cmap="inferno", vmin=Tw, vmax=FRT)
    ax.set_title(f"t = {tsnap:.1f} s")
fig.colorbar(im, ax=axes.ravel().tolist(), label="Temperature [degC]", shrink=0.9)
fig.suptitle("Thickness x Length temperature field (centre-width section)", fontsize=13)
fig.savefig(f"{FIGDIR}/contour_thickness_length.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True, sharey=True)
rows = [("Top surface", 0), ("Mid-thickness", i_c)]
for r, (rname, ii) in enumerate(rows):
    for c_, (tsnap, fld) in enumerate(snaps):
        ax = axes[r, c_]
        full = np.vstack([fld[ii][::-1], fld[ii]])   # mirror half width
        im = ax.imshow(full, extent=[0, L, -W / 2, W / 2], origin="lower", aspect="auto",
                       cmap="inferno", vmin=Tw, vmax=FRT)
        if r == 0: ax.set_title(f"t = {tsnap:.1f} s")
        if c_ == 0: ax.set_ylabel(rname, fontsize=10)
    ax.set_xlabel("Length [mm]")
fig.colorbar(im, ax=axes.ravel().tolist(), label="Temperature [degC]", shrink=0.8)
fig.suptitle("Width x Length temperature field (surface and mid-thickness planes)", fontsize=13)
fig.savefig(f"{FIGDIR}/contour_width_length.png", dpi=150)
plt.show()

## 10 · Cooling rates and the surface-to-core differential

Cooling rate $CR=-dT/dt$ at the four depths, and the differential $\Delta T = T_{surface}-T_{core}$, which drives residual stress, distortion and property gradients through the plate.

In [ ]:
CRs = -np.gradient(np.vstack([Ts_h, Tq_h, T3_h, Tc_h]), t_hist, axis=1)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t_hist, CRs[0], lw=2.5, color="tab:orange", label="Surface")
ax.plot(t_hist, CRs[1], lw=2,   color="tab:green",  label="Quarter depth")
ax.plot(t_hist, CRs[2], lw=2,   color="tab:purple", label="One-third depth")
ax.plot(t_hist, CRs[3], lw=2.5, color="k",          label="Core")
ax.axvline(t_acc_end, color="gray", ls="--", label="End of ACC")
ax.set_xlabel("Time [s]"); ax.set_ylabel("Cooling rate -dT/dt [degC/s]")
ax.set_title("Cooling rate at the four depths (plate centre)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{FIGDIR}/cooling_rate.png", dpi=150)
plt.show()

dT = Tc_h - Ts_h     # core minus surface: positive while the surface is colder
imax = int(np.argmax(dT))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t_hist, dT, lw=2, color="crimson")
ax.plot(t_hist[imax], dT[imax], "o", color="crimson",
        label=f"max dT = {dT[imax]:.0f} degC at t = {t_hist[imax]:.1f} s")
ax.axvline(t_acc_end, color="gray", ls="--", label="End of ACC")
ax.set_xlabel("Time [s]"); ax.set_ylabel("Core - Surface temperature [degC]")
ax.set_title("Through-thickness differential evolution"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIGDIR}/dt_surface_core.png", dpi=150)
plt.show()

## 11 · Watching the cooling sweep along the plate

Animation of the thickness × length plane. The hot core lingers long after the surfaces are quenched, and the cooled front sweeps from the leading edge to the trailing edge as the curtain train moves down the table.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))
im = ax.imshow(frames_T[0], extent=[0, L, t_h, 0], aspect="auto",
               cmap="inferno", vmin=Tw, vmax=FRT)
cb = fig.colorbar(im, ax=ax, label="Temperature [degC]")
ax.set_xlabel("Plate length [mm]"); ax.set_ylabel("Depth [mm]")
def update(i):
    im.set_data(frames_T[i])
    ax.set_title(f"t = {frames_t[i]:.2f} s")
    return (im,)
from matplotlib import animation
anim = animation.FuncAnimation(fig, update, frames=len(frames_t), interval=120)
plt.close(fig)
try:
    display(HTML(anim.to_html5_video()))
except Exception as exc:
    print("ffmpeg unavailable in this runtime, showing key frames instead:", exc)
    fig2, axes2 = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
    picks = np.linspace(0, len(frames_t) - 1, 4).astype(int)
    for ax2, p in zip(axes2, picks):
        ax2.imshow(frames_T[p], extent=[0, L, t_h, 0], aspect="auto",
                   cmap="inferno", vmin=Tw, vmax=FRT)
        ax2.set_title(f"t = {frames_t[p]:.2f} s")
    plt.show()

## 12 · Validation and summary

The FDM is benchmarked against the estimates from section 6: the lumped capacitance solution, the Heisler one-term approximation and the semi-infinite surface response. One caveat applies to both analytical models. They assume the water HTC acts continuously over the whole timeline, while the FDM applies it only during curtain contact, so the two values bracket the FDM result. Lumped sits below it, because continuous quenching over-predicts the cooling, and the semi-infinite solution matches the early surface behaviour.

In [ ]:
# =====================================================================
#  VALIDATION AND SUMMARY
# =====================================================================
Fo_end  = alpha * t_acc_end / (t_h_m / 2) ** 2
T_lump_end  = Tw + (FRT - Tw) * np.exp(-t_acc_end / tau_lump)
T_heis_end  = Tw + (FRT - Tw) * C1 * np.exp(-zeta1 ** 2 * Fo_end)

dT = Tc_h - Ts_h     # core minus surface: positive while the surface is colder
imax = int(np.argmax(dT))
T_final = {
    "Surface":            T[0, j_c, k_c],
    "Quarter depth (1/4)": T[i_q, j_c, k_c],
    "One-third depth (1/3)": T[i_t3, j_c, k_c],
    "Core":               T[i_c, j_c, k_c],
}
stability_ok = T.min() >= Tw - 1e-6
surface_faster = Ts_h[-1] < Tc_h[-1] or dT[imax] > 0

display(Markdown(f"""
### Final state (after ACC + {t_post:.0f} s air cooling, plate centre)

| Position | Final temperature [degC] |
|---|---|
| Surface ({x_t[0]*1e3:.1f} mm) | {T_final['Surface']:.1f} |
| Quarter depth ({x_t[i_q]*1e3:.1f} mm) | {T_final['Quarter depth (1/4)']:.1f} |
| One-third depth ({x_t[i_t3]*1e3:.1f} mm) | {T_final['One-third depth (1/3)']:.1f} |
| Core ({x_t[i_c]*1e3:.1f} mm) | {T_final['Core']:.1f} |

### Validation

| Check | Value | Verdict |
|---|---|---|
| Biot number | {Bi:.2f} | {'lumped valid' if Bi < 0.1 else 'FDM required (Bi > 0.1)'} |
| Tuning factor (water HTC scale) | {tune_factor:.2f} | {'calibrated to plant data' if tune_factor != 1.0 else 'uncalibrated (correlation as-is)'} |
| FDM core temperature at end of ACC | {Tc_h[np.argmin(np.abs(t_hist - t_acc_end))]:.1f} degC | n/a |
| Lumped-capacitance estimate at end of ACC | {T_lump_end:.1f} degC | {'agrees' if abs(T_lump_end - Tc_h[np.argmin(np.abs(t_hist - t_acc_end))]) < 50 else 'deviates (expected for Bi > 0.1)'} |
| Heisler one-term estimate at end of ACC | {T_heis_end:.1f} degC | benchmark |
| Semi-infinite surface after 1 curtain | {Ts_1hdr:.1f} degC | benchmark |
| Max surface-core differential | {dT[imax]:.1f} degC at t = {t_hist[imax]:.1f} s | n/a |
| Stability (min T >= water temp) | {T.min():.1f} degC | {'PASS' if stability_ok else 'FAIL'} |
| Surface cools faster than core | n/a | {'PASS' if surface_faster else 'FAIL'} |
"""))

## 13 · The story, in short

- The surface is quenched by every curtain passage, while the core barely notices a single one and follows only through conduction, so the differential peaks while the curtain train is still passing.
- Since $Bi \gg 0.1$, the lumped model is no use for design. The FDM resolves the quarter-depth and one-third-depth temperatures that sit between the two extremes.
- Width and length gradients come from the side edges and from the leading and trailing edges losing heat to air, which shows up as cold rims in every contour.
- Change `t_h, W, L, n_cur, v, FRT, s_gap` in the inputs cell and re-run. The whole pipeline, from the empirical pre-calculation through the FDM to the contours, updates by itself.
- When plant measurements disagree with the prediction, calibrate the `tune_factor` (section 6b) from a measured finish cooling temperature, and every downstream result carries the calibrated severity.

The transient thermal equations used here are in sections 1 and 2.